# 03 - Validate Neo4j Graph and Query Patterns

This notebook validates the loaded PrimeKG graph and captures query patterns that the FastAPI backend can reuse for the BioKG Explorer application.

## 1. Connect to Neo4j

Use the same environment variables as the loading notebook. Keep these variables aligned with the backend service configuration later.

In [14]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


## 2. Query Helper

Define a small helper that runs Cypher and returns pandas DataFrames. This keeps validation and API-prototype queries concise.

In [15]:
def run_cypher(query, parameters=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, parameters or {})
        return pd.DataFrame([record.data() for record in result])

## 3. Validate Node Counts by Label

Confirm that the database contains the expected node labels and counts. The `Entity` label should cover every PrimeKG node.

In [16]:
node_label_counts = run_cypher(
    """
    MATCH (n:Entity)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS count
    ORDER BY count DESC
    """
)
node_label_counts

,label,count
0,Entity,129375
1,PrimeKG,129375
2,BiologicalProcess,28642
3,GeneProtein,27671
4,Disease,17080
5,Phenotype,15311
6,Anatomy,14035
7,MolecularFunction,11169
8,Drug,7957
9,CellularComponent,4176


## 4. Validate Relationship Counts by Type

Confirm the distribution of Neo4j relationship types. These should correspond to the normalized PrimeKG relation names.

In [17]:
relationship_type_counts = run_cypher(
    """
    MATCH ()-[r]-()
    RETURN type(r) AS relationship_type, r.relation AS relation, r.display_relation AS display_relation, count(r) AS count
    ORDER BY count DESC
    """
)
relationship_type_counts

,relationship_type,relation,display_relation,count
0,ANATOMY_PROTEIN_PRESENT,anatomy_protein_present,expression present,3036406
1,DRUG_DRUG,drug_drug,synergistic interaction,2672628
2,PROTEIN_PROTEIN,protein_protein,ppi,642150
3,DISEASE_PHENOTYPE_POSITIVE,disease_phenotype_positive,phenotype present,300634
4,BIOPROCESS_PROTEIN,bioprocess_protein,interacts with,289610
5,CELLCOMP_PROTEIN,cellcomp_protein,interacts with,166804
6,DISEASE_PROTEIN,disease_protein,associated with,160822
7,MOLFUNC_PROTEIN,molfunc_protein,interacts with,139060
8,DRUG_EFFECT,drug_effect,side effect,129568
9,BIOPROCESS_BIOPROCESS,bioprocess_bioprocess,parent-child,105772


## 5. Full-Text Entity Search

Use Neo4j's full-text index to find nodes by name or descriptive text. This query pattern is a strong candidate for a FastAPI `/search` endpoint.

In [18]:
def search_entities(query_text, limit=20):
    return run_cypher(
        """
        CALL db.index.fulltext.queryNodes('entity_fulltext', $query_text) YIELD node, score
        RETURN
            node.primekg_index AS primekg_index,
            node.name AS name,
            node.node_type AS node_type,
            node.node_source AS node_source,
            score
        ORDER BY score DESC
        LIMIT $limit
        """,
        {"query_text": query_text, "limit": limit},
    )

search_entities("diabetes")

,primekg_index,name,node_type,node_source,score
0,38244,diabetes insipidus,disease,MONDO,13.057570
1,94794,lipoatrophic diabetes,disease,MONDO,13.057570
2,98053,monogenic diabetes,disease,MONDO,13.057570
3,84167,gestational diabetes,disease,MONDO,13.057570
4,99453,gestational diabetes insipidus,disease,MONDO,11.776531
5,99213,cardiomyopathy diabetes deafness,disease,MONDO,11.776531
6,84298,dipsogenic diabetes insipidus,disease,MONDO,11.776531
7,32446,nephrogenic diabetes insipidus,disease,MONDO,11.776531
8,28294,neurohypophyseal diabetes insipidus,disease,MONDO,11.776531
9,33575,diabetes mellitus (disease),disease,MONDO,11.776531


## 6. Entity Detail Query

Retrieve the full property map for a node. This query pattern can power detail drawers or side panels in the React frontend.

In [19]:
def get_entity_detail(primekg_index):
    result = run_cypher(
        """
        MATCH (n:Entity {primekg_index: $primekg_index})
        RETURN properties(n) AS properties, labels(n) AS labels
        """,
        {"primekg_index": int(primekg_index)},
    )
    return result.iloc[0].to_dict() if not result.empty else None

example = search_entities("diabetes", limit=1).iloc[0]
get_entity_detail(example["primekg_index"])

{'properties': {'disease_mayo_causes': "To understand diabetes, first you must understand how glucose is normally processed in the body. How insulin works, Insulin is a hormone that comes from a gland situated behind and below the stomach . The pancreas secretes insulin into the bloodstream. The insulin circulates, enabling sugar to enter your cells. Insulin lowers the amount of sugar in your bloodstream. As your blood sugar level drops, so does the secretion of insulin from your pancreas. The role of glucose, Glucose — a sugar — is a source of energy for the cells that make up muscles and other tissues. Glucose comes from two major sources: food and your liver. Sugar is absorbed into the bloodstream, where it enters cells with the help of insulin. Your liver stores and makes glucose. When your glucose levels are low, such as when you haven't eaten in a while, the liver breaks down stored glycogen into glucose to keep your glucose level within a normal range. Causes of type 1 diabetes,

## 7. Expand One-Hop Neighbors

Expand a node to view neighbors with optional node-type and relation filters. This is the core query pattern for interactive graph exploration in Cytoscape or D3.

In [20]:
def expand_neighbors(primekg_index, limit=100, node_types=None, relations=None):
    node_types = node_types or []
    relations = relations or []
    return run_cypher(
        """
        MATCH (source:Entity {primekg_index: $primekg_index})-[r]-(target:Entity)
        WHERE ($node_types = [] OR target.node_type IN $node_types)
          AND ($relations = [] OR r.relation IN $relations OR type(r) IN $relations)
        RETURN
            source.primekg_index AS source_index,
            source.name AS source_name,
            target.primekg_index AS target_index,
            target.name AS target_name,
            target.node_type AS target_type,
            type(r) AS relationship_type,
            r.relation AS relation,
            r.display_relation AS display_relation,
            properties(r) AS relationship_properties
        LIMIT $limit
        """,
        {
            "primekg_index": int(primekg_index),
            "limit": int(limit),
            "node_types": node_types,
            "relations": relations,
        },
    )

expand_neighbors(example["primekg_index"], limit=25)

,source_index,source_name,target_index,target_name,target_type,relationship_type,relation,display_relation,relationship_properties
0,38244,diabetes insipidus,16867,Demeclocycline,drug,CONTRAINDICATION,contraindication,contraindication,"{'undirected': True, 'primekg_row_count': 2, '..."
1,38244,diabetes insipidus,17829,Lithium citrate,drug,CONTRAINDICATION,contraindication,contraindication,"{'undirected': True, 'primekg_row_count': 2, '..."
2,38244,diabetes insipidus,17831,Lithium carbonate,drug,CONTRAINDICATION,contraindication,contraindication,"{'undirected': True, 'primekg_row_count': 2, '..."
3,38244,diabetes insipidus,20422,Meclocycline,drug,CONTRAINDICATION,contraindication,contraindication,"{'undirected': True, 'primekg_row_count': 2, '..."
4,38244,diabetes insipidus,84298,dipsogenic diabetes insipidus,disease,DISEASE_DISEASE,disease_disease,parent-child,"{'undirected': True, 'primekg_row_count': 2, '..."
5,38244,diabetes insipidus,99453,gestational diabetes insipidus,disease,DISEASE_DISEASE,disease_disease,parent-child,"{'undirected': True, 'primekg_row_count': 2, '..."
6,38244,diabetes insipidus,28294,neurohypophyseal diabetes insipidus,disease,DISEASE_DISEASE,disease_disease,parent-child,"{'undirected': True, 'primekg_row_count': 2, '..."
7,38244,diabetes insipidus,32446,nephrogenic diabetes insipidus,disease,DISEASE_DISEASE,disease_disease,parent-child,"{'undirected': True, 'primekg_row_count': 2, '..."
8,38244,diabetes insipidus,35764,kidney disease,disease,DISEASE_DISEASE,disease_disease,parent-child,"{'undirected': True, 'primekg_row_count': 2, '..."
9,38244,diabetes insipidus,1760,POMC,gene/protein,DISEASE_PROTEIN,disease_protein,associated with,"{'undirected': True, 'primekg_row_count': 2, '..."


## 8. Frontend Graph Payload Shape

Convert one-hop results into a simple `{nodes, edges}` payload that maps naturally to Cytoscape.js and D3 force-directed graph inputs.

In [21]:
def neighborhood_payload(primekg_index, limit=100, node_types=None, relations=None):
    rows = expand_neighbors(primekg_index, limit=limit, node_types=node_types, relations=relations)
    center = get_entity_detail(primekg_index)

    nodes_by_id = {
        int(primekg_index): {
            "id": str(primekg_index),
            "primekg_index": int(primekg_index),
            "label": center["properties"].get("name"),
            "node_type": center["properties"].get("node_type"),
        }
    }
    edges = []

    for row in rows.to_dict(orient="records"):
        target_id = int(row["target_index"])
        nodes_by_id[target_id] = {
            "id": str(target_id),
            "primekg_index": target_id,
            "label": row["target_name"],
            "node_type": row["target_type"],
        }
        edges.append(
            {
                "id": f"{primekg_index}-{target_id}-{row['relationship_type']}",
                "source": str(primekg_index),
                "target": str(target_id),
                "label": row["display_relation"] or row["relation"],
                "relationship_type": row["relationship_type"],
                "relation": row["relation"],
                "properties": row["relationship_properties"],
            }
        )

    return {"nodes": list(nodes_by_id.values()), "edges": edges}

payload = neighborhood_payload(example["primekg_index"], limit=10)
payload

{'nodes': [{'id': '38244',
   'primekg_index': 38244,
   'label': 'diabetes insipidus',
   'node_type': 'disease'},
  {'id': '16867',
   'primekg_index': 16867,
   'label': 'Demeclocycline',
   'node_type': 'drug'},
  {'id': '17829',
   'primekg_index': 17829,
   'label': 'Lithium citrate',
   'node_type': 'drug'},
  {'id': '17831',
   'primekg_index': 17831,
   'label': 'Lithium carbonate',
   'node_type': 'drug'},
  {'id': '20422',
   'primekg_index': 20422,
   'label': 'Meclocycline',
   'node_type': 'drug'},
  {'id': '84298',
   'primekg_index': 84298,
   'label': 'dipsogenic diabetes insipidus',
   'node_type': 'disease'},
  {'id': '99453',
   'primekg_index': 99453,
   'label': 'gestational diabetes insipidus',
   'node_type': 'disease'},
  {'id': '28294',
   'primekg_index': 28294,
   'label': 'neurohypophyseal diabetes insipidus',
   'node_type': 'disease'},
  {'id': '32446',
   'primekg_index': 32446,
   'label': 'nephrogenic diabetes insipidus',
   'node_type': 'disease'},
  

## 9. Shortest Path Query

Find a bounded shortest path between two entities. The maximum hop count is intentionally bounded so the query remains safe for interactive use.

In [22]:
def shortest_path_between(source_index, target_index, max_hops=5):
    max_hops = int(max_hops)
    if max_hops < 1 or max_hops > 10:
        raise ValueError("max_hops must be between 1 and 10 for interactive use")

    query = f"""
    MATCH (source:Entity {{primekg_index: $source_index}}),
          (target:Entity {{primekg_index: $target_index}})
    MATCH path = shortestPath((source)-[*..{max_hops}]-(target))
    RETURN
        [node IN nodes(path) | node {{ .primekg_index, .name, .node_type, .node_source }}] AS nodes,
        [rel IN relationships(path) | {{
            relationship_type: type(rel),
            relation: rel.relation,
            display_relation: rel.display_relation,
            properties: properties(rel)
        }}] AS relationships,
        length(path) AS hops
    """
    return run_cypher(query, {"source_index": int(source_index), "target_index": int(target_index)})

# Example: choose two search results before running.
search_entities("insulin", limit=5)

,primekg_index,name,node_type,node_source,score
0,15385,Insulin peglispro,drug,DrugBank,11.926702
1,116704,insulin binding,molecular_function,GO,11.926702
2,128190,Insulin processing,pathway,REACTOME,11.926702
3,21515,Insulin argine,drug,DrugBank,11.926702
4,97715,acute insulin response,disease,MONDO,11.774309


## 10. Relationship Detail Query

Retrieve relationship details between two nodes. This can support edge-click interactions in the graph UI.

In [23]:
def relationship_details(source_index, target_index):
    return run_cypher(
        """
        MATCH (source:Entity {primekg_index: $source_index})-[r]-(target:Entity {primekg_index: $target_index})
        RETURN
            source { .primekg_index, .name, .node_type } AS source,
            target { .primekg_index, .name, .node_type } AS target,
            type(r) AS relationship_type,
            properties(r) AS properties
        ORDER BY relationship_type
        """,
        {"source_index": int(source_index), "target_index": int(target_index)},
    )

neighbors = expand_neighbors(example["primekg_index"], limit=1)
if not neighbors.empty:
    relationship_details(example["primekg_index"], neighbors.iloc[0]["target_index"])
else:
    print("No neighbors found for example node")

## 12. Close the Driver

Close the database connection after validation.

In [24]:
driver.close()
print("Neo4j driver closed")

Neo4j driver closed
